In [1]:
import pandas as pd
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from pprint import pprint
import torch
from google.colab import files
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('hugging_face'))

Reading in csv

In [2]:
url = "https://raw.githubusercontent.com/bethancunningham/nlp_2026/main/dataset_assignment3.csv"
df = pd.read_csv(url)

out_path = 'results.csv'

Models

In [3]:
models = ['goldfish-models/cym_latn_10mb', 'britllm/britllm-3b-v0.1', 'meta-llama/Llama-3.1-8B']

Creating function to calculate NLL (based on Rodríguez & Brochhagen 2026)

In [4]:
def calculate_sequence_nll(model, tokenizer, sequence: str) -> float:
    """
    Function to calculate NLL of sequence using a causal language model.
    Lower NLL indicates higher probability.

    """
    try:
        # Tokenising input sequence (using tokens like <bos> (beginning of sequence) if the model uses them)
        inputs = tokenizer(sequence, return_tensors='pt', add_special_tokens=True)

        # Moving inputs to the same device as the model
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Getting model outputs (logits) - calculates loss against input_ids to get per-token NLLs
        with torch.no_grad(): # Disabling gradient calculation for inference
            outputs = model(**inputs, labels=inputs['input_ids'])

        # Getting average NLL across all tokens (loss returned by AutoModelForCausalLM. Lower loss = more likely
        nll = outputs.loss.item()
        return nll

    except Exception as e:
        print(f"Error calculating NLL for sequence '{sequence}': {e}")
        return float('inf') # Returning infinity for errors

Creating function to compare sentence likelihoods (based on Rodríguez & Brochhagen 2026)

In [5]:
def compare_sequence_likelihoods(model, tokenizer, sequence1: str, sequence2: str):
    """
    Function to compare the likelihood of two sequences using their NLL
    """
    print(f"Comparing likelihoods using model: {model.config._name_or_path}")
    nll1 = calculate_sequence_nll(model, tokenizer, sequence1)
    nll2 = calculate_sequence_nll(model, tokenizer, sequence2)

    print(f"\nSequence 1: '{sequence1}'")
    print(f"  Negative Log-Likelihood (NLL): {nll1:.4f}")

    print(f"Sequence 2: '{sequence2}'")
    print(f"  Negative Log-Likelihood (NLL): {nll2:.4f}")

    if nll1 < nll2:
        print(f"\nConclusion: Sequence 1 is more likely than Sequence 2 (Lower NLL).")
    elif nll2 < nll1:
        print(f"\nConclusion: Sequence 2 is more likely than Sequence 1 (Lower NLL).")
    else:
        print(f"\nConclusion: Both sequences have similar likelihoods.")
    return([nll1,nll2])


Iterating over each model, iterating over each row, calculating NLLs for pairs of sentences and comparing them

In [ ]:
for model_name in models:
  print(f'...processing {model_name}')
  # Loading model
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
  # Creating list for results per row
  outs = []
  # Iterating over rows, getting NLL for each sentence
  for i, (index, row) in enumerate(df.iterrows()):
    print(f'... ... current index: {index}')
    p1, p2 = compare_sequence_likelihoods(model, tokenizer, row['sentence'], row['incorrect_sentence'])
    new_row = {
      'index': index,
      'model': model_name,
      'sentence_id': row['sent_id'],
      'sentence_en': row['sentence_en'],
      'correct_sentence':  row['sentence'],
      'correct_word': row['correct_word'],
      'mutation_type': row['mutation_type'],
      'trigger_type': row['trigger_type'],
      'incorrect_word': row['incorrect_word'],
      'incorrect_sentence': row['incorrect_sentence'],
      'p_correct': p1,
      'p_incorrect': p2
    }
    outs.append(new_row)

    # Saving and downloading every 50 rows
    if (i + 1) % 50 == 0:
      checkpoint_path = f'checkpoint_{model_name.replace("/", "_")}_{i + 1}.csv'
      pd.DataFrame(outs).to_csv(checkpoint_path, index=False)
      files.download(checkpoint_path)
      print(f'... ... checkpoint saved at row {i + 1}')

  # Final save for any remaining rows after the loop
  if outs:
    model_safe_name = model_name.replace("/", "_")
    final_path = f'final_{model_safe_name}.csv'
    df_out = pd.DataFrame(outs)
    df_out.to_csv(final_path, index=False)
    files.download(final_path)

...processing goldfish-models/cym_latn_10mb


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Filtering last 50 rows for Llama because processing stopped at 290
df_remaining = df.iloc[250:]

model_name = 'meta-llama/Llama-3.1-8B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

outs = []

for i, (index, row) in enumerate(df_remaining.iterrows()):
  print(f'... ... current index: {index}')
  p1, p2 = compare_sequence_likelihoods(model, tokenizer, row['sentence'], row['incorrect_sentence'])
  new_row = {
    'index': index,
    'model': model_name,
    'sentence_id': row['sent_id'],
    'sentence_en': row['sentence_en'],
    'correct_sentence': row['sentence'],
    'correct_word': row['correct_word'],
    'mutation_type': row['mutation_type'],
    'trigger_type': row['trigger_type'],
    'incorrect_word': row['incorrect_word'],
    'incorrect_sentence': row['incorrect_sentence'],
    'p_correct': p1,
    'p_incorrect': p2
  }
  outs.append(new_row)

df_out = pd.DataFrame(outs)
df_out.to_csv('llama_final_50.csv', index=False)
files.download('llama_final_50.csv')